In [ ]:
# Model Benchmarking

## Objective
'''
Compare classical ML baseline with Transformer-based models.

Models:
1. TF-IDF + Logistic Regression
2. DistilBERT
3. RoBERTa-base
'''

In [1]:
from datasets import load_from_disk
from google.colab import drive
drive.mount('/drive')

dataset_split = load_from_disk(
    "/drive/MyDrive/dataset_split"
)

tokenized_dataset = load_from_disk(
    "/drive/MyDrive/tokenized_dataset"
)

Mounted at /drive


In [2]:
X_train = dataset_split["train"]["text"]

y_train = dataset_split["train"]["label"]

X_val = dataset_split["validation"]["text"]

y_val = dataset_split["validation"]["label"]

**Expriment 1**

TF-IDF + Logistic Regression

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000
    )),

    ("clf", LogisticRegression(
        max_iter=1000
    ))
])

In [8]:
baseline_model.fit(
    X_train,
    y_train
)

Pipeline(steps=[('tfidf', TfidfVectorizer(max_features=20000)),
                ('clf', LogisticRegression(max_iter=1000))])

In [9]:
y_pred = baseline_model.predict(
    X_val
)

In [16]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd


accuracy = accuracy_score(
    y_val,
    y_pred
)

macro_f1 = f1_score(
    y_val,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_val,
    y_pred,
    average="weighted"
)

result = pd.DataFrame( )
result["metrics"] = ["accuracy", "macro_f1", "weighted_f1"]
result["value"] = [accuracy, macro_f1, weighted_f1]
result.set_index("metrics", inplace=True)
display(result)


,value
metrics,
accuracy,0.868132
macro_f1,0.860975
weighted_f1,0.865336


'print(accuracy)\nprint(macro_f1)\nprint(weighted_f1)'

Experiment 2

DistilBERT

In [3]:
checkpoint = "distilbert-base-uncased"

from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [4]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True
    )

tokenized_distilbert = dataset_split.map(
    tokenize_function
)



Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

In [5]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=77
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [7]:
from sklearn.metrics import accuracy_score, f1_score


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = logits.argmax(axis=-1)

    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

In [8]:
from transformers import TrainingArguments


training_args = TrainingArguments(
    output_dir="./distilbert_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    report_to="none"
)

In [9]:
from transformers import Trainer


trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_distilbert["train"],

    eval_dataset=tokenized_distilbert["validation"],

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,3.398585,2.057543,0.696304,0.629216,0.657508
2,1.793374,1.131938,0.823177,0.783925,0.810157
3,1.143987,0.922074,0.854146,0.824669,0.845789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1689, training_loss=1.9863214487151468, metrics={'train_runtime': 134.123, 'train_samples_per_second': 201.352, 'train_steps_per_second': 12.593, 'total_flos': 278940662865972.0, 'train_loss': 1.9863214487151468, 'epoch': 3.0})

In [10]:

from transformers import AutoTokenizer

checkpoint = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [11]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True
    )

tokenized_roberta = dataset_split.map(
    tokenize_function
)



Map:   0%|          | 0/9002 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

In [12]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [13]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=77
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
from sklearn.metrics import accuracy_score, f1_score


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = logits.argmax(axis=-1)

    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

In [15]:
from transformers import TrainingArguments


training_args = TrainingArguments(
    output_dir="./roberta_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    report_to="none"
)

In [16]:
from transformers import Trainer


trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_roberta["train"],

    eval_dataset=tokenized_roberta["validation"],

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,2.887219,1.199934,0.820180,0.784703,0.802825
2,1.094978,0.595441,0.902098,0.898477,0.900799
3,0.623742,0.488176,0.907093,0.906801,0.906172


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1689, training_loss=1.422545338749815, metrics={'train_runtime': 256.6611, 'train_samples_per_second': 105.22, 'train_steps_per_second': 6.581, 'total_flos': 541576695911100.0, 'train_loss': 1.422545338749815, 'epoch': 3.0})